![Henry Logo](https://www.soyhenry.com/_next/static/media/HenryLogo.bb57fd6f.svg)

# M3L2 E00 - ChatOpenAI: el modelo como objeto (Resolution)

## BLOQUE 3 — Herramienta 1: ChatOpenAI (LLM Wrapper)

### Qué es y para qué sirve

`ChatOpenAI` encapsula el modelo. El resto del pipeline **no necesita saber** cómo funciona la API de OpenAI.

## Configuración inicial: API key

In [ ]:
import os, getpass
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

### Sin LangChain (mostrar en vivo o en pizarrón)

```python
from openai import OpenAI
client = OpenAI()
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Hola"}],
    temperature=0,
)
texto = response.choices[0].message.content
```

> *"Fíjense: el modelo está hardcodeado acá adentro. Si en 10 funciones distintas llaman así al modelo y quieren cambiar de gpt-4o-mini a gpt-4o, tienen que buscar en 10 lugares."*

In [ ]:
from openai import OpenAI

client = OpenAI()
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Di solo la palabra: hola"}],
    temperature=0,
)
texto = response.choices[0].message.content
print(f"Sin LangChain - tipo: {type(texto).__name__}, valor: {texto}")

### Con LangChain

```python
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
result = llm.invoke("Hola")
print(result.content)
```

> *"El modelo queda en un objeto. El resto del código lo recibe como parámetro. Si quiero cambiar el modelo, cambio UNA línea."*

#### TODO 1: Crear el objeto ChatOpenAI

- **Objetivo**: crear un `ChatOpenAI` con `model="gpt-4o-mini"` y `temperature=0`.
- **Lo que demuestra**: el modelo es un objeto configurable que se puede pasar como parámetro.

In [ ]:
from langchain_openai import ChatOpenAI

# TODO 1: crear ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print(f"Tipo: {type(llm).__name__}, modelo: {llm.model_name}")

### Detalle importante: devuelve AIMessage, no string

```python
result = llm.invoke("Di 'hola'")
print(type(result))        # AIMessage
print(result.content)      # el texto
```

> *"LangChain devuelve un AIMessage porque necesita guardar metadatos junto con el texto: cuántos tokens usó, ID del mensaje, etc. Para obtener el texto, accedemos a .content."*

#### TODO 2: Invocar con `.invoke()` y ver que devuelve AIMessage

- **Objetivo**: llamar a `llm.invoke("Di solo la palabra: hola")` e inspeccionar el tipo.
- **Lo que demuestra**: `invoke()` devuelve `AIMessage`, no un string crudo.

In [ ]:
# TODO 2: invocar y observar el tipo
result = llm.invoke("Di solo la palabra: hola")
print(f"Tipo de result: {type(result).__name__}")
print(f"result.content: {result.content}")
print()
print("ChatOpenAI devuelve un AIMessage, no un string.")
print("Para el texto: result.content")

#### TODO 3: Crear dos configuraciones con temperaturas distintas y comparar respuestas

- **Objetivo**: crear `llm_preciso` (temp=0) y `llm_creativo` (temp=1.0) y pasarles la misma pregunta.
- **Lo que demuestra**: el modelo es un objeto configurable. Temperatura baja = respuestas determinísticas; temperatura alta = más variedad.

In [ ]:
# TODO 3: dos configuraciones
llm_preciso = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_creativo = ChatOpenAI(model="gpt-4o-mini", temperature=1.0)

pregunta = "Describe el cielo en 5 palabras"
print(f"Temperatura 0:   {llm_preciso.invoke(pregunta).content}")
print(f"Temperatura 1.0: {llm_creativo.invoke(pregunta).content}")

## Resumen — Lo que demuestra E00

| Sin LangChain | Con LangChain |
|---|---|
| `response.choices[0].message.content` | `result.content` (más limpio) |
| Modelo hardcodeado en cada llamada | Modelo en un objeto configurable |
| Formato específico de OpenAI | Interfaz estándar `.invoke()` |
| No se puede pasar como componente | Se pasa como objeto a una chain |

**El modelo es un objeto configurable, no una función hardcodeada.**

## Ejemplos con otros proveedores

LangChain wrappea **cualquier** proveedor con la misma interfaz `.invoke()`. El resto del pipeline no cambia — ni el prompt, ni el parser, ni la chain.

| Proveedor | Clase de LangChain | Paquete | Variable de entorno |
|---|---|---|---|
| OpenAI | `ChatOpenAI` | `langchain-openai` | `OPENAI_API_KEY` |
| Anthropic (Claude) | `ChatAnthropic` | `langchain-anthropic` | `ANTHROPIC_API_KEY` |
| Google (Gemini) | `ChatGoogleGenerativeAI` | `langchain-google-genai` | `GOOGLE_API_KEY` |
| Ollama (modelo local, sin costo ni API key) | `ChatOllama` | `langchain-ollama` | — (corre en tu máquina) |

Las celdas de abajo son **opcionales**: cada una se saltea sola con un mensaje si falta el paquete, la API key, o (en el caso de Ollama) si no tenés el servidor corriendo. No hace falta tener las 4 configuradas para seguir el notebook.

In [1]:
# Anthropic (Claude)
# pip install langchain-anthropic  |  requiere ANTHROPIC_API_KEY

try:
    from langchain_anthropic import ChatAnthropic

    if os.getenv("ANTHROPIC_API_KEY"):
        llm_claude = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0)
        resultado_claude = llm_claude.invoke("Di solo la palabra: hola")
        print(f"Claude responde: {resultado_claude.content}")
        print(f"Mismo tipo que ChatOpenAI: {type(resultado_claude).__name__}")
    else:
        print("Salteado: falta ANTHROPIC_API_KEY en el entorno.")
except ImportError:
    print("Salteado: falta instalar langchain-anthropic (pip install langchain-anthropic).")

Salteado: falta instalar langchain-anthropic (pip install langchain-anthropic).


In [ ]:
# Google (Gemini)
# pip install langchain-google-genai  |  requiere GOOGLE_API_KEY

try:
    from langchain_google_genai import ChatGoogleGenerativeAI

    if os.getenv("GOOGLE_API_KEY"):
        llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
        resultado_gemini = llm_gemini.invoke("Di solo la palabra: hola")
        print(f"Gemini responde: {resultado_gemini.content}")
        print(f"Mismo tipo que ChatOpenAI: {type(resultado_gemini).__name__}")
    else:
        print("Salteado: falta GOOGLE_API_KEY en el entorno.")
except ImportError:
    print("Salteado: falta instalar langchain-google-genai (pip install langchain-google-genai).")

In [ ]:
# Ollama (modelo local, corre en tu maquina, sin costo ni API key)
# 1) Instalar Ollama: https://ollama.com
# 2) Descargar un modelo:  ollama pull llama3.2
# 3) pip install langchain-ollama

try:
    from langchain_ollama import ChatOllama

    llm_local = ChatOllama(model="llama3.2", temperature=0)
    resultado_local = llm_local.invoke("Di solo la palabra: hola")
    print(f"Ollama (local) responde: {resultado_local.content}")
    print(f"Mismo tipo que ChatOpenAI: {type(resultado_local).__name__}")
except ImportError:
    print("Salteado: falta instalar langchain-ollama (pip install langchain-ollama).")
except Exception as e:
    print(f"Salteado: no se pudo conectar a Ollama local. Verifica que 'ollama serve' este corriendo.")
    print(f"Detalle: {e}")

In [ ]:
# El mismo prompt, en todos los proveedores que hayan quedado disponibles en esta sesion

proveedores = {"OpenAI (gpt-4o-mini)": llm}
if globals().get("llm_claude"):
    proveedores["Anthropic (Claude)"] = llm_claude
if globals().get("llm_gemini"):
    proveedores["Google (Gemini)"] = llm_gemini
if globals().get("llm_local"):
    proveedores["Ollama (local)"] = llm_local

print(f"Proveedores disponibles en esta sesion: {list(proveedores.keys())}")
print()

pregunta_bonus = "Describe el cielo en 5 palabras"
for nombre, modelo in proveedores.items():
    respuesta = modelo.invoke(pregunta_bonus)
    print(f"{nombre}: {respuesta.content}")

print()
print("La unica linea que cambia entre proveedores es la que crea el objeto llm.")
print("El prompt, el parser y la chain siguen siendo exactamente el mismo codigo.")

## ¿Por qué conviene usar LangChain y no el SDK directo?

### Comparación punto por punto

| Situación | SDK directo | LangChain |
|---|---|---|
| **Cambiar de proveedor** | Reescribes todo el código que toca el LLM | Cambias 1 línea: `ChatOpenAI` → `ChatAnthropic` |
| **Agregar memoria** | Tú mismo guardas el historial, lo serializas, lo inyectas en cada llamada | Usas `RunnableWithMessageHistory` y ya |
| **Conectar tools** | Parseás el JSON de `tool_calls` a mano, invocás, devolvés resultado | Decorás con `@tool`, usás `bind_tools()` |
| **Hacer RAG** | Implementás embeddings, vector store, retrieval, prompt armado | `Chroma + retriever + LCEL chain` |
| **Encadenar llamadas** | Anidás promesas/callbacks o escribes glue code | Usás el pipe `prompt | llm | parser` |
| **Estandarizar respuestas** | Cada proveedor devuelve un objeto distinto | Siempre `AIMessage.content` |
| **Testear sin red** | Tenés que mockear la API completa | Usás `FakeListChatModel` de LangChain |
| **Producción (tracing, monitoreo)** | Lo implementás desde cero | LangSmith, callbacks, metadatos incluidos |

### El argumento principal

> *"LangChain no te da una API para un modelo. Te da una **capa de abstracción** que funciona igual para todos los modelos, todos los vectostores, todos los parsers. Tu código se escribe una vez y se adapta sin reescribir."*

### ¿Y si solo uso un proveedor y no pienso cambiar?

Igual conviene porque el día que necesites **memoria, tools, RAG, chains**, LangChain ya lo tiene resuelto. Hacerlo con el SDK directo implica implementar cada pieza desde cero o copiar soluciones de internet que probablemente ya existen en LangChain.

In [ ]:
def run_checks():
    from langchain_core.messages import AIMessage
    assert isinstance(llm, ChatOpenAI)
    result = llm.invoke("Di solo: test")
    assert isinstance(result, AIMessage)
    assert len(result.content) > 0
    print("M3L2 E00 Resolution checks passed")

run_checks()